# CERRA experiment workflow

This notebook turns **one settings cell** into everything a run needs:

1. **check** that every referenced `.nc` file exists (train/val/test splits, static files, masks, steady mask),
2. **write the JSON file lists** under `<ml_path>/data/` (`data/{train,val,test}/<var>/*.json`, `data/masks/<var>/`, `data/test/masks/<var>/`),
3. **write the train and eval config files** (`--flag value` text, loaded via `--load-from-file`) into `input/levante/`, named `train-cnn-<config-name>-<run-name>.txt` / `eval-cnn-<config-name>-<run-name>.txt`.

Experiments are identified by their `--config-name` / `--run-name` pair — training and evaluation write their outputs into `snapshots/`, `logs/` and `evaluations/` under exactly that pair, so the pair must stay unique. The workflow enforces this:

- rerunning the **same** experiment (identical generated configs) is idempotent — `action: unchanged`, nothing new is written,
- a **changed** experiment reusing the same names gets a `-r2`, `-r3`, ... suffix (`action: suffixed`), or raises with `on_duplicate = "error"`,
- existing output directories (`snapshots/`, `evaluations/`, `logs/` under `<ml_path>`) are also taken into account.

Edit the settings cell below, then **Run All**. The last cell prints the `sbatch` commands to submit training and per-year evaluation.

In [ ]:
import os
import sys

# ---- cluster paths --------------------------------------------------------
# ml_path: checkout of this repo on the cluster (JSONs go to <ml_path>/data/,
#          configs to <ml_path>/input/levante/, outputs to <ml_path>/{snapshots,logs,evaluations}/)
ml_path = "/work/uo1075/u244085/trees_in_the_city/ML_downscaling/Physics-Informed-CNN-Reconstructions"
if not os.path.isdir(ml_path):
    ml_path = os.path.abspath(os.path.join(os.getcwd(), os.pardir))

# data_root: CERRA data on the cluster (per-variable folders + mask/)
data_root = "/work/uo1075/u244085/cerra_data"

if os.path.isdir(os.path.join(ml_path, "climatereconstructionai")) and ml_path not in sys.path:
    sys.path.insert(0, ml_path)

# ---- file-name layout (suffixes used in the .nc / .json names) -------------
reg = "ger"                              # region suffix, e.g. _ger
other = "remapcon_land"                  # target/static remap suffix, e.g. _remapcon_land
res = "2deg"                             # input resolution suffix, e.g. _2deg
descriptor = "_lowres_values_filled_highres_grid"
static_year = 2005                       # year of the single static files (orography, lsm)
steady_mask_year = 1985                  # year of the single steady mask

splits = {
    "train": (1985, 2005),
    "val": (2006, 2015),
    "test": (2016, 2025),
}

# ---- experiment configurations ---------------------------------------------
# key = experiment tag, "target" defaults to the key, "static_vars" to []
CONFIGS = {
    "relative_humidity": {
        "input_vars": ["relative_humidity", "t2m", "surface_pressure", "wind_speed", "wind_direction"],
        "static_vars": ["orography"],
    },
    "tp_v2": {
        "target": "tp",
        "input_vars": ["tciwv", "t2m", "relative_humidity", "surface_pressure", "wind_speed", "wind_direction"],
        "static_vars": ["orography"],
    },
    "tp": {
        "target": "tp",
        "input_vars": ["tp", "t2m", "relative_humidity", "surface_pressure", "wind_speed", "wind_direction"],
        "static_vars": ["orography"],
    },
    "t2m": {
        "target": "t2m",
        "input_vars": ["t2m", "relative_humidity", "surface_pressure", "wind_speed", "wind_direction"],
        "static_vars": ["orography"],
    },
    "t2m_single": {"target": "t2m", "input_vars": ["t2m"], "static_vars": []},
    "relative_humidity_single": {"target": "relative_humidity", "input_vars": ["relative_humidity"], "static_vars": []},
    "tp_single": {"target": "tp", "input_vars": ["tp"], "static_vars": []},
}

# ---- one entry per experiment to generate -----------------------------------
# run_name (+ optional config_name) identifies the experiment and must be unique
# per config_name: rerunning an unchanged experiment is idempotent, a changed one
# gets a -rN suffix. train_params/eval_params override DEFAULT_MODEL_PARAMS
# (eval_params defaults to train_params); model_names defaults to "best".
EXPERIMENTS = {
    "tp_v2": {
        "config_name": "tp",
        "run_name": "cerra-germany-0.05x2deg-el5-all-var-l6-bilin-cfill",
        "train_params": {"lr": "5e-5", "conv-factor": 512, "loss-criterion": 6},
    },
}

selected = "tp_v2"          # one key of EXPERIMENTS, or None for all
config_dir = os.path.join(ml_path, "input", "levante")
on_duplicate = "suffix"    # "suffix": append -rN on name clash; "error": raise instead

## Build the workflow from the settings

Resolves the `.nc` / `.json` name templates and the eval file templates (`{var}`, `{year}`, `{reg}` stay literal for the evaluate slurm script).


In [ ]:
from climatereconstructionai.utils.config_generator import CerraConfigWorkflow

wf = CerraConfigWorkflow(
    data_root=data_root,
    ml_path=ml_path,
    reg=reg,
    other=other,
    res=res,
    descriptor=descriptor,
    static_year=static_year,
    steady_mask_year=steady_mask_year,
    splits=splits,
    configs=CONFIGS,
)

print("input        :", wf.nc_tpl["input"])
print("target       :", wf.nc_tpl["target"])
print("static       :", wf.nc_tpl["static"])
print("mask         :", wf.nc_tpl["mask"])
print("steady mask  :", wf.nc_tpl["steady_mask"])
print()
print("eval --data-file-template   :", wf.eval_tpl["data"])
print("eval --target-file-template :", wf.eval_tpl["target"])
print("eval --steady-mask-file     :", wf.eval_tpl["steady_mask"])

## 1. Check that all referenced files exist

Fails early (and writes nothing) if any `.nc` file of the selected configs is missing.


In [ ]:
missing = wf.check_all(selected=selected, include_test=True)
if missing:
    total = sum(len(m) for m in missing.values())
    print(f"ERROR: {total} file(s) missing -- nothing will be written:")
    for key, files in missing.items():
        print(f"  [{key}] {len(files)} missing:")
        for f in files:
            print(f"    {f}")
    raise SystemExit(1)
print("all files present")

## 2. Write JSON lists + unique train/eval configs

Writes the JSON file lists under `<ml_path>/data/` and the two config files into `input/levante/`. Rerunning an unchanged experiment reports `unchanged`; a changed one gets a `-rN` run-name suffix.


In [ ]:
experiments = {k: v for k, v in EXPERIMENTS.items() if selected is None or k == selected}
results = wf.run_all(experiments, config_dir=config_dir, on_duplicate=on_duplicate, include_test=True)

for r in results:
    print(f"[{r['action']:>9}] config-name={r['config_name']}  run-name={r['run_name']}")
    print(f"  train config : {r['train_file']}")
    print(f"  eval  config : {r['eval_file']}")
    print(f"  json lists   : {len(r['json_files'])} file(s) under {os.path.join(ml_path, 'data')}")
    print(f"  submit train : {r['train_cmd']}")
    print(f"  submit eval  : {r['eval_cmd']}")
    print()

## Submitting the runs

From the repo root on the cluster (`ml_path`):

```bash
# training (config file name as printed above)
sbatch input/slurm-scripts/slurm-train-crai.sh train-cnn-<config-name>-<run-name>.txt

# per-year evaluation over the test split (slurm-eval-crai.sh resolves
# {var}/{reg}/{year} in the --*-file-template lines for each year)
CONFIG_FILE=eval-cnn-<config-name>-<run-name>.txt sbatch input/slurm-scripts/slurm-eval-crai.sh

# analysis of the evaluated years (figures, netCDF, stats)
input/slurm-scripts/4a_submit_evaluation.sh --config eval-cnn-<config-name>-<run-name>.txt
```

Outputs land in `snapshots/<config-name>/<run-name>/`, `logs/<config-name>/<run-name>/` and `evaluations/<config-name>/<run-name>/`.